# providers.azure

OpenAI chat completions served from an Azure deployment.

In [ ]:
#| default_exp providers.azure

In [ ]:
#| hide
from nbdev.showdoc import *

## The provider

Construction is cheap and side-effect-free — credentials and config are resolved lazily inside `_get_client`. That keeps `from nbdialog.providers.azure import AzureProvider` safe in environments without credentials (offline doc builds, CI without secrets, tests that mock `complete`).

`complete` is a request/response loop, not a single call. When `tools` are passed in, the model can answer directly *or* emit a `tool_calls` array — in which case we dispatch each call against the registered Python function, append the result as a `role: "tool"` message, and re-call. `max_tool_steps` bounds the loop in case the model gets stuck. When no tools are passed, the loop exits on the first response.

Endpoint and deployment have no defaults — set them via the environment or pass them explicitly:

```python
# option 1: env vars (recommended)
# export AZURE_OPENAI_ENDPOINT=https://your-resource.cognitiveservices.azure.com
# export AZURE_OPENAI_DEPLOYMENT=your-deployment-name
# export AZURE_API_KEY=...
set_provider(AzureProvider())

# option 2: explicit
set_provider(AzureProvider(endpoint="https://...", deployment="gpt-5.4"))
```

In [ ]:
#| export
import json, os, time
from openai import AzureOpenAI
from nbdialog.core import Tool, Trace

class AzureProvider:
    "OpenAI chat completions via an Azure deployment, with a tool-call loop and optional Trace recording."
    def __init__(self,
                 deployment: str | None = None,
                 endpoint: str | None = None,
                 api_version: str = "2024-12-01-preview",
                 api_key_env: str = "AZURE_API_KEY",
                 max_completion_tokens: int = 16384,
                 max_tool_steps: int = 8):
        self.deployment = deployment or os.environ.get("AZURE_OPENAI_DEPLOYMENT")
        self.endpoint = endpoint or os.environ.get("AZURE_OPENAI_ENDPOINT")
        self.api_version, self.api_key_env = api_version, api_key_env
        self.max_completion_tokens = max_completion_tokens
        self.max_tool_steps = max_tool_steps
        self._client = None

    def _get_client(self):
        if self._client is None:
            if not self.endpoint:
                raise RuntimeError("AzureProvider: no endpoint. Set $AZURE_OPENAI_ENDPOINT or pass endpoint=...")
            if not self.deployment:
                raise RuntimeError("AzureProvider: no deployment. Set $AZURE_OPENAI_DEPLOYMENT or pass deployment=...")
            self._client = AzureOpenAI(api_key=os.environ[self.api_key_env],
                                       azure_endpoint=self.endpoint,
                                       api_version=self.api_version)
        return self._client

    def complete(self, messages: list[dict], tools: list[Tool] = None,
                 trace: Trace = None) -> str:
        tools = tools or []
        schemas = [t.schema for t in tools]
        dispatch = {t.schema["function"]["name"]: t.fn for t in tools}
        msgs = list(messages)
        for _ in range(self.max_tool_steps):
            kw = {"tools": schemas} if schemas else {}
            t0 = time.perf_counter()
            resp = self._get_client().chat.completions.create(
                model=self.deployment, messages=msgs,
                max_completion_tokens=self.max_completion_tokens, **kw)
            dt = time.perf_counter() - t0
            m = resp.choices[0].message
            if trace is not None:
                tc_for_trace = [{"name": tc.function.name,
                                 "args": json.loads(tc.function.arguments)}
                                for tc in (m.tool_calls or [])]
                usage = resp.usage.model_dump() if resp.usage else None
                trace.add_model_turn(m.content, tc_for_trace, usage, dt)
            if not m.tool_calls: return m.content
            msgs.append(m.model_dump(exclude_none=True))
            for tc in m.tool_calls:
                args = json.loads(tc.function.arguments)
                t0 = time.perf_counter()
                out = dispatch[tc.function.name](**args)
                dt = time.perf_counter() - t0
                if trace is not None:
                    trace.add_tool_turn(tc.function.name, args, str(out), dt)
                msgs.append({"role": "tool", "tool_call_id": tc.id,
                             "content": str(out)})
        raise RuntimeError(f"Tool loop did not converge in {self.max_tool_steps} steps")

Smoke test — only runs when credentials are present, so the doc build stays green without them.

In [ ]:
if all(os.environ.get(k) for k in ("AZURE_API_KEY", "AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_DEPLOYMENT")):
    print(AzureProvider().complete([{"role":"user","content":"ping"}]))

pong


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()